# 🧪 01 – Data Preprocessing
**Proyek:** AI-Based Pharmaceutical Data Selection & Monitoring System  
**Tim:** PJK-GM016 | Pijak × IBM SkillsBuild  
**Minggu:** 2 – Data Cleaning, Preprocessing & Feature Engineering

---
### Alur Notebook
| Langkah | Deskripsi |
|---------|-----------|
| 1 | Load data CSV mentah |
| 2 | Hapus duplikat |
| 3 | Filter baris valid |
| 4 | Konversi kolom numerik |
| 5 | Analisis missing values |
| 6 | Imputasi missing values |
| 7 | Parsing kadar air |
| 8 | Parsing tanggal |
| 9 | Feature engineering |
| 10 | Ringkasan statistik |
| 11 | Simpan output bersih |


## ⚙️ Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.1)
print("✅ Library berhasil diimport")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")

## 📂 Step 1 – Load Data

In [ ]:
INPUT_CSV  = "rekap_produksi_granulasi_raw.csv"
OUTPUT_CSV = "rekap_produksi_clean.csv"

df = pd.read_csv(INPUT_CSV)
print(f"Shape awal   : {df.shape}")
print(f"Jumlah kolom : {df.shape[1]}")
print(f"Jumlah baris : {df.shape[0]}")
df.head(3)

## 🗑️ Step 2 – Hapus Duplikasi

In [ ]:
before = len(df)
df = df.drop_duplicates()
after  = len(df)
print(f"Baris sebelum  : {before}")
print(f"Baris setelah  : {after}")
print(f"Duplikat dihapus: {before - after}")

## 🔍 Step 3 – Filter Baris Valid (ada Material)

In [ ]:
before = len(df)
df = df.dropna(subset=['Material_Description', 'Material_Code'])
after  = len(df)
print(f"Baris dihapus (tidak ada material): {before - after}")
print(f"Baris valid: {after}")
df['Material_Description'].value_counts()

## 🔢 Step 4 – Konversi Kolom Numerik

In [ ]:
NUMERIC_COLS = [
    'GB_Yield_Kg_L1', 'GB_Yield_Kg_L2',
    'GK_Yield_Kg_L1', 'GK_Yield_Kg_L2',
    'Cetak_Yield_Kg', 'Cetak_Pct_Teoritis',
    'Kemas_Yield_Blister', 'Kemas_Pct_Teoritis',
    'Tablet_Rusak_Kg', 'Sampah_Cacahan_Kg',
    'Blister_Kosong_Kg', 'Waste_Sedotan_Kg',
    'Waste_Sample_IPC_Kg', 'Kemul_Tambahan'
]

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("✅ Konversi selesai")
df[NUMERIC_COLS].dtypes

## ❓ Step 5 – Analisis Missing Values

In [ ]:
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
mv_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
mv_df = mv_df[mv_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

# Visualisasi
fig, ax = plt.subplots(figsize=(10, 6))
top_mv = mv_df.head(20)
bars = ax.barh(top_mv.index, top_mv['Missing %'], color='#e74c3c', edgecolor='white')
ax.set_xlabel("Missing (%)")
ax.set_title("Top 20 Kolom dengan Missing Values", fontweight='bold')
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.show()

mv_df

## 🩹 Step 6 – Imputasi Missing Values

In [ ]:
# Kolom waste / reject → isi 0 (tidak ada waste = 0)
waste_cols = [
    'Tablet_Rusak_Kg', 'Sampah_Cacahan_Kg', 'Blister_Kosong_Kg',
    'Waste_Sedotan_Kg', 'Waste_Sample_IPC_Kg', 'Kemul_Tambahan',
    'Hasil_Blister_Reject', 'Sealing', 'Forming'
]
for col in waste_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Kolom yield → isi dengan median per material
yield_cols = [
    'GB_Yield_Kg_L1', 'GB_Yield_Kg_L2',
    'GK_Yield_Kg_L1', 'GK_Yield_Kg_L2',
    'Cetak_Yield_Kg', 'Kemas_Yield_Blister'
]
for col in yield_cols:
    if col in df.columns:
        df[col] = df.groupby('Material_Code')[col].transform(
            lambda x: x.fillna(x.median())
        )

# Kolom % teoritis → isi dengan median per material
pct_cols = ['Cetak_Pct_Teoritis', 'Kemas_Pct_Teoritis']
for col in pct_cols:
    if col in df.columns:
        df[col] = df.groupby('Material_Code')[col].transform(
            lambda x: x.fillna(x.median())
        )

remaining_mv = df.isnull().sum()
remaining_mv = remaining_mv[remaining_mv > 0]
print("Missing values tersisa:")
print(remaining_mv if len(remaining_mv) else "✅ Tidak ada missing value di kolom kunci")

## 💧 Step 7 – Parsing Kadar Air
> Format asli: `'5,50/5,46/5,52'` → diambil rata-rata

In [ ]:
def parse_kadar_air(val):
    """Ekstrak rata-rata dari string format '5.40/5.49/5.59'."""
    if pd.isna(val):
        return np.nan
    val = str(val).replace(',', '.').replace('H', '').strip()
    parts = val.split('/')
    nums = []
    for p in parts:
        try:
            nums.append(float(p.strip()))
        except Exception:
            pass
    return np.mean(nums) if nums else np.nan

if 'GB_Kadar_Air_L1' in df.columns:
    df['GB_Kadar_Air_L1_num'] = df['GB_Kadar_Air_L1'].apply(parse_kadar_air)

if 'GB_Kadar_Air_L2' in df.columns:
    df['GB_Kadar_Air_L2_num'] = df['GB_Kadar_Air_L2'].apply(parse_kadar_air)

# Preview
sample = df[['GB_Kadar_Air_L1', 'GB_Kadar_Air_L1_num']].dropna()
print("Contoh hasil parsing kadar air:")
sample.head(8)

## 📅 Step 8 – Konversi Kolom Tanggal

In [ ]:
date_cols = [
    'GB_Tanggal_CB_L1', 'GB_Tanggal_CB_L2',
    'GK_Tanggal_CK_L1', 'GK_Tanggal_CK_L2',
    'Cetak_Tgl_Mulai', 'Cetak_Tgl_Selesai',
    'Kemas_Tgl_Mulai', 'Kemas_Tgl_Selesai'
]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print("✅ Kolom tanggal berhasil dikonversi")
df[[c for c in date_cols if c in df.columns]].dtypes

## 🛠️ Step 9 – Feature Engineering
> Membuat fitur-fitur baru yang relevan untuk model AI

In [ ]:
# 9a. Total Yield Granulasi Basah
df['GB_Yield_Total'] = df['GB_Yield_Kg_L1'].fillna(0) + df['GB_Yield_Kg_L2'].fillna(0)

# 9b. Total Yield Granulasi Kering
df['GK_Yield_Total'] = df['GK_Yield_Kg_L1'].fillna(0) + df['GK_Yield_Kg_L2'].fillna(0)

# 9c. Rasio efisiensi pengeringan (GK/GB)
df['Rasio_GK_GB'] = np.where(
    df['GB_Yield_Total'] > 0,
    df['GK_Yield_Total'] / df['GB_Yield_Total'],
    np.nan
)

# 9d. Total waste (semua sumber)
df['Total_Waste_Kg'] = (
    df['Tablet_Rusak_Kg'].fillna(0) +
    df['Sampah_Cacahan_Kg'].fillna(0) +
    df['Blister_Kosong_Kg'].fillna(0) +
    df['Waste_Sedotan_Kg'].fillna(0) +
    df['Waste_Sample_IPC_Kg'].fillna(0)
)

# 9e. Label defect (threshold % teoritis < 90%)
THRESHOLD = 0.90
df['Defect_Cetak']   = (df['Cetak_Pct_Teoritis']  < THRESHOLD).astype(int)
df['Defect_Kemas']   = (df['Kemas_Pct_Teoritis']  < THRESHOLD).astype(int)
df['Defect_Overall'] = ((df['Defect_Cetak'] == 1) | (df['Defect_Kemas'] == 1)).astype(int)

# 9f. Durasi proses (hari)
if 'Cetak_Tgl_Mulai' in df.columns and 'Cetak_Tgl_Selesai' in df.columns:
    df['Cetak_Durasi_Hari'] = (df['Cetak_Tgl_Selesai'] - df['Cetak_Tgl_Mulai']).dt.days.clip(lower=0)

if 'Kemas_Tgl_Mulai' in df.columns and 'Kemas_Tgl_Selesai' in df.columns:
    df['Kemas_Durasi_Hari'] = (df['Kemas_Tgl_Selesai'] - df['Kemas_Tgl_Mulai']).dt.days.clip(lower=0)

# 9g. Bulan & tahun produksi
if 'Cetak_Tgl_Mulai' in df.columns:
    df['Bulan_Produksi'] = df['Cetak_Tgl_Mulai'].dt.month
    df['Tahun_Produksi'] = df['Cetak_Tgl_Mulai'].dt.year

# 9h. Rata-rata kadar air
if 'GB_Kadar_Air_L1_num' in df.columns:
    df['GB_Kadar_Air_Mean'] = df[['GB_Kadar_Air_L1_num', 'GB_Kadar_Air_L2_num']].mean(axis=1, skipna=True)

new_feats = [
    'GB_Yield_Total', 'GK_Yield_Total', 'Rasio_GK_GB', 'Total_Waste_Kg',
    'Defect_Cetak', 'Defect_Kemas', 'Defect_Overall',
    'Cetak_Durasi_Hari', 'Kemas_Durasi_Hari',
    'Bulan_Produksi', 'Tahun_Produksi', 'GB_Kadar_Air_Mean'
]
print("✅ Fitur baru berhasil dibuat:")
for f in new_feats:
    if f in df.columns:
        non_null = df[f].notna().sum()
        print(f"   {f:30s} → {non_null} nilai valid")

## 📊 Step 10 – Ringkasan Statistik

In [ ]:
key_cols = [
    'GB_Yield_Total', 'GK_Yield_Total', 'Rasio_GK_GB',
    'Cetak_Yield_Kg', 'Cetak_Pct_Teoritis',
    'Kemas_Pct_Teoritis', 'Total_Waste_Kg',
    'Defect_Overall', 'GB_Kadar_Air_Mean'
]
key_cols = [c for c in key_cols if c in df.columns]
df[key_cols].describe().round(4)

In [ ]:
# Distribusi label
print("Distribusi Defect_Overall:")
vc = df['Defect_Overall'].value_counts()
for k, v in vc.items():
    label = 'Normal' if k == 0 else 'Defect'
    pct = v / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"  {label:8s} ({k}) : {v:4d} baris  {pct:5.1f}%  {bar}")

## 💾 Step 11 – Simpan Data Bersih

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ File tersimpan  : {OUTPUT_CSV}")
print(f"   Shape final    : {df.shape}")
print(f"   Total fitur    : {df.shape[1]}")